In [ ]:
import scanpy as sc
import logging
import rpy2.rinterface_lib.callbacks as rcb
import anndata2ri

rcb.logger.setLevel(logging.ERROR)

%load_ext rpy2.ipython
anndata2ri.activate()


In [2]:
adata = sc.read("../results/adata/05-filter-ambient.h5ad")

In [3]:
mat = adata.X.T
samples = adata.obs["sample"]

# Detect Doublets

In [4]:
%%R -i mat -i samples -o doublet_score -o doublet_class

suppressMessages(library(scDblFinder))

set.seed(1234)
sce = SingleCellExperiment(list(counts=mat))
colData(sce)$samples <- samples
sce = scDblFinder(sce, samples="samples", clusters=TRUE)
doublet_score = sce$scDblFinder.score
doublet_class = sce$scDblFinder.class

  |======================================================================| 100%



In [5]:
adata.obs["scdblfinder_score"] = doublet_score
adata.obs["scdblfinder_class"] = doublet_class

In [6]:
adata.obs["scdblfinder_class"].value_counts()

scdblfinder_class
1    31721
2     2143
Name: count, dtype: int64

In [7]:
adata.write("../results/adata/06-doublets.h5ad")